# 📝 그래프 집계 과제 LV1(기초): 집계·그룹핑·collect·CASE·인덱스

> 이 단원의 새 기술을 **하나씩** 확인합니다. count·sum·avg·min/max·collect, WITH 집계 후 필터, SET 파생 속성, CASE 라벨(단순형), 인덱스 생성.

## 풀이 방법
1. 맨 위 **준비 셀 3개**(연결 → 초기화 → 시드 적재)를 먼저 실행하세요. 반드시 **실습 전용 DB**로.
2. 각 문제의 **답안 셀**에 코드를 채우고 **자가채점 셀**로 확인하세요(✅ 통과!).
3. Cypher 는 교안처럼 `run_cypher("...쿼리...")` 로 실행하고, 결과(dict 리스트)를 지정한 변수에 담으세요.

- 도메인: **음식 배달**: 고객(Customer)이 메뉴(Menu)를 주문(ORDERED{qty,price})하고, 식당(Restaurant)이 메뉴를 제공(SERVES)합니다.

화이팅!

아래 준비 셀이 만들 그래프의 전체 모습입니다. 식당 3곳·메뉴 7개·고객 4명이 두 종류의 관계로 이어져 있습니다. **주문이 한 번도 없는 메뉴가 하나** 있는 것에 주목하세요(9·13번에서 다룹니다).

<img src="images/그래프_한눈에_배달.png" width="820">

아래 준비 셀 3개를 위에서부터 실행하세요.

In [ ]:
# [제공 코드] Neo4j 연결: 실행만 하세요. .env 로 연결하고 run_cypher 헬퍼를 만듭니다.
# 반드시 "실습 전용" DB 에 연결하세요. 아래 실습이 그래프를 지우고 새로 만듭니다.
import os

from dotenv import load_dotenv
from neo4j import GraphDatabase
from neo4j.exceptions import ConstraintError  # UNIQUE 제약 위반 에러

# 1) 접속 정보: .env 를 환경변수로 올린다(비밀번호를 코드에 적지 않으려고)
load_dotenv(".env")       # 같은 폴더의 .env
load_dotenv("../.env")    # 정답 폴더에서 실행하는 경우

# .env 를 못 읽어도 에러가 아니라 기본값으로 넘어간다. 마지막 줄의 주소를 눈으로 확인할 것
NEO4J_URI = os.getenv("NEO4J_URI", "bolt://localhost:7687")
NEO4J_USER = os.getenv("NEO4J_USER", "neo4j")
NEO4J_PASSWORD = os.getenv("NEO4J_PASSWORD", "neo4j")

# 2) 드라이버: 노트북이 끝날 때까지 재사용할 통로 하나
driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))
driver.verify_connectivity()   # 여기서 에러가 나면 주소나 계정이 틀린 것


# 3) 공용 헬퍼: 앞으로 모든 Cypher 는 이 함수 하나로 실행한다
def run_cypher(query, **params):
    """Cypher 실행 -> 결과를 dict 리스트로 반환(수업 공용 헬퍼)."""
    with driver.session() as session:
        # record.data() 가 한 행을 dict 로 바꾼다. 키는 RETURN 의 별칭이다
        return [record.data() for record in session.run(query, **params)]


print("Neo4j 연결:", NEO4J_URI)

In [ ]:
# [제공 코드] 그래프 초기화: 실습 전용 DB 인지 꼭 확인하고 실행하세요!
# 노드·관계에 더해 이 노트북이 만든 인덱스·제약까지 지웁니다(DB 기본 LOOKUP 인덱스는 그대로).
# 1) 제약 먼저. 제약이 남아 있으면 그 제약이 만든 인덱스를 따로 못 지운다
for _row in run_cypher("SHOW CONSTRAINTS YIELD name RETURN name"):
    run_cypher("DROP CONSTRAINT " + _row["name"] + " IF EXISTS")
# 2) 남은 인덱스. LOOKUP 은 DB 기본이라 뺀다
for _row in run_cypher("SHOW INDEXES YIELD name, type WHERE type <> 'LOOKUP' RETURN name"):
    run_cypher("DROP INDEX " + _row["name"] + " IF EXISTS")
# 3) 노드·관계. 관계가 9만 개라 한 번에 담지 않고 2만 개씩 끊어 지운다
while True:
    # DETACH DELETE 는 매달린 관계까지 함께 지운다
    _left = run_cypher("MATCH (n) WITH n LIMIT 20000 DETACH DELETE n RETURN count(n) AS n")[0]["n"]
    if _left == 0:
        break
print("초기화 완료: 남은 노드:", run_cypher("MATCH (n) RETURN count(n) AS n")[0]["n"])

In [ ]:
# [제공 코드] 음식 배달 시드 적재: 실행만 하세요.
# Restaurant{category} -SERVES-> Menu{price} <-ORDERED{qty, price}- Customer
menus = [
    ["비빔밥", 9000, "한마당", "한식"],
    ["불고기", 12000, "한마당", "한식"],
    ["연어초밥", 15000, "초밥왕", "일식"],
    ["우동", 8000, "초밥왕", "일식"],
    ["까르보나라", 13000, "파스타팩토리", "양식"],
    ["마르게리타", 14000, "파스타팩토리", "양식"],
    ["알리오올리오", 11000, "파스타팩토리", "양식"],
]
customers = ["강민", "서연", "준호", "예린"]
orders = [
    ["강민", "비빔밥", 2, 18000], ["강민", "연어초밥", 1, 15000],
    ["서연", "연어초밥", 2, 30000], ["서연", "까르보나라", 1, 13000],
    ["준호", "비빔밥", 1, 9000], ["준호", "불고기", 2, 24000], ["준호", "연어초밥", 1, 15000],
    ["예린", "마르게리타", 2, 28000], ["예린", "우동", 1, 8000], ["예린", "까르보나라", 1, 13000],
]
for name, price, restaurant, category in menus:
    run_cypher("MERGE (rt:Restaurant {name: $restaurant}) SET rt.category = $category "
               "MERGE (m:Menu {name: $name}) SET m.price = $price "
               "MERGE (rt)-[:SERVES]->(m)", restaurant=restaurant, category=category, name=name, price=price)
for name in customers:
    run_cypher("MERGE (:Customer {name: $name})", name=name)
for customer, menu, qty, price in orders:
    run_cypher("MATCH (c:Customer {name: $customer}), (m:Menu {name: $menu}) "
               "MERGE (c)-[x:ORDERED]->(m) SET x.qty = $qty, x.price = $price",
               customer=customer, menu=menu, qty=qty, price=price)
print("적재 완료: 메뉴:", run_cypher("MATCH (m:Menu) RETURN count(m) AS n")[0]["n"],
      "· 주문:", run_cypher("MATCH ()-[x:ORDERED]->() RETURN count(x) AS n")[0]["n"])


## 그래프 살펴보기

문제로 들어가기 전에 **무엇이 들어 있는지** 한 번 훑습니다. 어떤 레이블이 몇 개인지, 어떤 관계가 몇 개이고 거기에 어떤 속성이 붙어 있는지를 알아야 "무엇을 무엇으로 묶을지"를 정할 수 있습니다.

In [ ]:
# [제공 코드] 그래프에 무엇이 들어 있는지 훑어봅니다(실행만 하세요).
# 1) 레이블마다 몇 개인지. 노드마다 레이블이 하나뿐이라 labels(n)[0] 로 충분하다
for row in run_cypher("MATCH (n) RETURN labels(n)[0] AS 레이블, count(*) AS 개수 ORDER BY 레이블"):
    print(row)

# 2) 관계 타입마다 몇 개인지. type(x) 가 관계 종류 이름이다
for row in run_cypher("MATCH ()-[x]->() RETURN type(x) AS 관계, count(x) AS 개수 ORDER BY 관계"):
    print(row)

# 3) 관계에 붙은 속성 이름. keys(x) 가 그 목록이다
for row in run_cypher("MATCH ()-[x]->() "
                      "RETURN type(x) AS 관계, collect(DISTINCT keys(x)) AS 속성키 ORDER BY 관계"):
    print(row)


## 1. 전체 메뉴 수 세기
**배경**: 등록된 메뉴가 모두 몇 개인지 셉니다.

**요구사항**:
- 모든 `Menu` 노드의 개수를 세어 `run_cypher` 결과를 변수 **`rows`** 에 담으세요. 별칭은 **`메뉴수`**.

**예시**: 결과는 `rows[0]['메뉴수']` 가 **7** 입니다.

<details><summary>힌트</summary>

```text
접근방법:
- 모든 Menu 노드를 잡아 count 로 센다(그룹핑 항이 없으면 전체가 한 줄로 세어진다).

세부구현:
1. MATCH 로 Menu 를 잡는다.
2. RETURN 에 count 집계를 요구사항의 별칭(메뉴수)으로 둔다.
3. run_cypher 결과를 rows 에 담는다.
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
assert rows[0]['메뉴수'] == 7, '전체 메뉴 수가 다릅니다. count 로 Menu 노드를 셌는지 확인하세요'
print('✅ 통과!')

## 2. 식당별 메뉴 수 (그룹핑)
**배경**: 식당마다 몇 개의 메뉴를 제공하는지 셉니다. 비집계 항이 **그룹핑 키**가 됩니다.

**요구사항**:
- `(rt:Restaurant)-[:SERVES]->(m:Menu)` 에서 식당별 메뉴 수를 세어 **`rows`** 에 담으세요. 별칭은 **`식당`**·**`메뉴수`**.
- **메뉴수 내림차순, 같으면 식당 이름 오름차순**으로 정렬하세요.

**예시**: `rows[0]` 은 식당 **파스타팩토리**, 메뉴수 **3** 입니다(가장 많음).

<details><summary>힌트</summary>

```text
접근방법:
- 식당과 메뉴를 SERVES 로 잇고, 식당 이름을 그룹핑 키로 두어 메뉴를 센다.

세부구현:
1. MATCH 로 식당→메뉴(SERVES) 패턴을 잡는다.
2. RETURN 에 식당 이름(그룹핑 키)과 count 집계를 별칭(식당·메뉴수)으로 둔다.
3. ORDER BY 로 메뉴수 내림차순, 같으면 식당 이름 오름차순으로 정렬한다.
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
assert rows[0]['식당'] == '파스타팩토리' and rows[0]['메뉴수'] == 3, '1위 식당이 다릅니다. 식당 이름을 그룹핑 키로 두었는지 확인하세요'
assert rows == sorted(rows, key=lambda r: (-r['메뉴수'], r['식당'])), '정렬을 확인하세요: 메뉴수 내림차순, 같으면 식당 이름 오름차순'
print('✅ 통과!')

## 3. 메뉴별 총 주문 수량 (sum)
**배경**: 메뉴마다 주문 수량(`qty`)의 합을 구해 가장 많이 나간 메뉴를 봅니다.

**요구사항**:
- `(:Customer)-[x:ORDERED]->(m:Menu)` 에서 메뉴별 `sum(x.qty)` 를 구해 **`rows`** 에 담으세요. 별칭은 **`메뉴`**·**`총수량`**.
- **총수량 내림차순, 같으면 메뉴 이름 오름차순**으로 정렬하세요.

**예시**: `rows[0]` 은 메뉴 **연어초밥**, 총수량 **4** 입니다. 결과는 **6행**이고 총수량을 모두 더하면 **14** 입니다.

<details><summary>힌트</summary>

```text
접근방법:
- 고객→메뉴(ORDERED) 를 잇고 메뉴 이름을 그룹핑 키로 두어, 주문 관계의 qty 를 sum 으로 합한다.

세부구현:
1. MATCH 로 고객→메뉴(ORDERED) 패턴을 잡고, 주문 관계에 변수를 붙인다.
2. RETURN 에 메뉴 이름과 sum 집계를 별칭(메뉴·총수량)으로 둔다.
3. ORDER BY 로 총수량 내림차순, 같으면 메뉴 이름 오름차순으로 정렬한다.
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
assert len(rows) == 6, '6행이어야 합니다. 주문이 있는 메뉴만 세지는지 확인하세요'
assert rows[0]['메뉴'] == '연어초밥' and rows[0]['총수량'] == 4, '1위가 다릅니다. 건수(count)가 아니라 수량 합(sum)을 냈는지 확인하세요'
assert rows == sorted(rows, key=lambda r: (-r['총수량'], r['메뉴'])), '정렬을 확인하세요: 총수량 내림차순, 같으면 메뉴 이름 오름차순'
assert sum(r['총수량'] for r in rows) == 14, \
    '총수량을 모두 더한 값이 다릅니다. 건수(count)가 아니라 수량 합(sum)을 냈는지 확인하세요'
print('✅ 통과!')

## 4. 고객별 평균 결제금액 (avg)
**배경**: 고객마다 주문 결제금액(`price`)의 평균을 구합니다.

**요구사항**:
- `(c:Customer)-[x:ORDERED]->(:Menu)` 에서 고객별 `avg(x.price)` 를 구해 **`rows`** 에 담으세요. 별칭은 **`고객`**·**`평균결제`**.
- **고객 이름 오름차순**으로 정렬하세요.

**예시**: 고객 **서연**의 `평균결제` 는 **21500.0** 입니다.

<details><summary>힌트</summary>

```text
접근방법:
- 고객 이름을 그룹핑 키로 두고 주문 관계의 price 를 avg 로 평균 낸다.

세부구현:
1. MATCH 로 고객→메뉴(ORDERED) 패턴을 잡고 주문 관계에 변수를 붙인다.
2. RETURN 에 고객 이름과 avg 집계를 별칭(고객·평균결제)으로 둔다.
3. ORDER BY 로 고객 이름 오름차순 정렬한다.
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
assert len(rows) == 4, '고객 4명이 모두 나와야 합니다'
seoyeon = [r for r in rows if r['고객'] == '서연'][0]
assert abs(seoyeon['평균결제'] - 21500.0) < 0.5, \
    '서연의 평균결제가 다릅니다. 결제금액은 관계의 price 속성이고, 합이 아니라 avg 입니다'
assert rows == sorted(rows, key=lambda r: r['고객']), '고객 이름 오름차순으로 정렬하세요'
print('✅ 통과!')

## 5. 가장 비싼·가장 싼 메뉴 가격 (min/max)
**배경**: 전체 메뉴 중 최고가와 최저가를 한 번에 구합니다.

**요구사항**:
- 모든 `Menu` 의 `price` 로 최고가·최저가를 구해 **`rows`** 에 담으세요. 별칭은 **`최고가`**·**`최저가`**.

**예시**: `rows[0]['최고가']` 는 **15000**, `rows[0]['최저가']` 는 **8000** 입니다.

<details><summary>힌트</summary>

```text
접근방법:
- 모든 메뉴를 잡고, 가격의 max 와 min 을 한 RETURN 에 나란히 둔다.

세부구현:
1. MATCH 로 Menu 를 잡는다.
2. RETURN 에 가격의 max·min 두 집계를 별칭(최고가·최저가)으로 나란히 둔다.
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
assert rows[0]['최고가'] == 15000 and rows[0]['최저가'] == 8000, 'max·min 으로 메뉴 가격의 최대·최소를 냈는지 확인하세요'
print('✅ 통과!')

## 6. 식당의 메뉴 목록 (collect)
**배경**: 파스타팩토리가 제공하는 메뉴 이름을 **리스트**로 모읍니다.

**요구사항**:
- 식당 **`파스타팩토리`** 가 제공하는 메뉴 이름을 `collect` 로 모아 **`rows`** 에 담으세요. 별칭은 **`메뉴목록`**.

**예시**: `rows[0]['메뉴목록']` 을 정렬하면 **['까르보나라', '마르게리타', '알리오올리오']** 입니다(리스트 순서는 상관없습니다).

<details><summary>힌트</summary>

```text
접근방법:
- 식당 이름을 파스타팩토리로 못박아 필터한 뒤(패턴의 노드 속성), 메뉴 이름을 collect 로 모은다.

세부구현:
1. MATCH 로 식당(이름=파스타팩토리)→메뉴(SERVES) 패턴을 잡는다.
2. RETURN 에 메뉴 이름을 collect 한 집계를 별칭(메뉴목록)으로 둔다.
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
assert len(rows) == 1, '한 행이어야 합니다. 비집계 항이 없으면 전체가 한 묶음이 됩니다'
assert sorted(rows[0]['메뉴목록']) == ['까르보나라', '마르게리타', '알리오올리오'], '메뉴 목록이 다릅니다. 파스타팩토리로 좁혔는지, collect 로 이름을 모았는지 확인하세요'
print('✅ 통과!')

## 7. 식당별 메뉴 목록을 모았다가 다시 펼치기 (collect + UNWIND)
**배경**: 6번에서 본 것처럼 `collect` 는 여러 행을 리스트 한 칸으로 접습니다. 접어 둔 목록을 다시 한 줄씩 다뤄야 할 때 쓰는 것이 `UNWIND` 입니다.

**요구사항**:
- `(rt:Restaurant)-[:SERVES]->(m:Menu)` 를 식당별로 묶어 메뉴 이름을 `collect` 로 모으세요.
- 그 리스트를 `UNWIND` 로 펼쳐 **`rows`** 에 담으세요. 별칭은 **`식당`**·**`메뉴`**.
- **식당 이름 오름차순, 같으면 메뉴 이름 오름차순**으로 정렬하세요.

**예시**: 펼친 결과는 **7행**으로 전체 메뉴 수와 같습니다(접었다 펴면 원래 행 수로 돌아옵니다). 첫 행은 `초밥왕`의 `연어초밥` 입니다.

<details><summary>힌트</summary>

```text
접근방법:
- 식당을 그룹핑 키로 두고 메뉴 이름을 리스트로 모은 뒤, 그 리스트를 한 줄씩 펼친다.
- 집계한 값을 다음 단계에서 쓰려면 RETURN 이 아니라 WITH 로 넘겨야 한다.

세부구현:
1. MATCH 로 식당→메뉴(SERVES) 패턴을 잡는다.
2. WITH 에 식당 이름과 메뉴 이름을 모은 리스트를 별칭과 함께 적는다.
3. UNWIND 로 그 리스트를 한 줄씩 펼치고 별칭을 메뉴로 둔다.
4. RETURN 에 식당·메뉴를 두고 식당 오름차순, 같으면 메뉴 오름차순으로 정렬한다.
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
assert len(rows) == 7, '펼친 행 수가 전체 메뉴 수(7)와 같아야 합니다. collect 로 모은 리스트를 UNWIND 로 펼쳤는지 확인하세요'
assert sorted(r['메뉴'] for r in rows) == ['까르보나라', '마르게리타', '불고기', '비빔밥', '알리오올리오', '연어초밥', '우동'], '메뉴가 빠지거나 중복됐습니다. UNWIND 대상이 collect 로 모은 리스트가 맞는지 확인하세요'
assert rows == sorted(rows, key=lambda r: (r['식당'], r['메뉴'])), '정렬을 확인하세요: 식당 이름 오름차순, 같으면 메뉴 이름 오름차순'
# DB 로 검산: 펼친 행 수는 SERVES 관계 수와 같아야 한다
serves = run_cypher("MATCH (:Restaurant)-[x:SERVES]->(:Menu) RETURN count(x) AS n")[0]['n']
assert len(rows) == serves, '펼친 행 수가 식당-메뉴 관계 수와 다릅니다'
print('✅ 통과!')

## 8. 주문이 2건 이상인 메뉴 (WITH 필터와 `COUNT { }`)
**배경**: 주문 **행이 2건 이상**인 메뉴만 골라 인기 메뉴를 봅니다. "주문 건수 2 이상"은 **집계 결과에 대한 조건**이라 `WITH` 로 먼저 집계한 뒤 걸러야 합니다. 같은 답을 `COUNT { }` 서브쿼리로 짧게 낼 수도 있습니다(교안_01 4-2). 두 길을 다 써 보고 답이 같은지 봅니다.

**요구사항**:
- **(1)** `(:Customer)-[x:ORDERED]->(m:Menu)` 를 메뉴별로 `count(x)` 집계한 뒤, **주문 건수 2 이상**만 남겨 **`rows`** 에 담으세요. 별칭은 **`메뉴`**·**`주문건수`**. **주문건수 내림차순, 같으면 메뉴 이름 오름차순**으로 정렬하세요.
- **(2)** 같은 답을 **`COUNT { }` 서브쿼리**로 한 번 더 내세요. `MATCH (m:Menu)` 만 잡고 `WHERE` 에 `COUNT { }` 조건을 걸어 **메뉴 이름 오름차순**으로 조회한 뒤, **메뉴 이름만 담은 문자열 리스트**를 **`sub_names`** 에 담으세요.

**예시**: (1) 조건을 만족하는 메뉴는 **3개**이고, `rows[0]` 은 **연어초밥**(주문건수 **3**)입니다. (2) `sub_names` 는 (1)과 같은 메뉴 **3개**를 이름순으로 담은 리스트입니다.

<details><summary>힌트</summary>

```text
접근방법:
- 집계 조건이므로 WITH 로 메뉴별 주문 건수(count)를 먼저 낸 뒤, 그 다음 WHERE 로 거른다.

세부구현:
1. MATCH 로 고객→메뉴(ORDERED) 를 잡고 주문 관계에 변수를 붙인다.
2. WITH 로 메뉴와 count 집계(별칭 주문건수)를 넘긴다.
3. 이어지는 WHERE 로 주문건수 2 이상만 남긴다.
4. RETURN 에 메뉴 이름·주문건수를 두고, 주문건수 내림차순(같으면 메뉴순)으로 ORDER BY 한다.
5. (2) MATCH (m:Menu) 만 잡고, WHERE 에 COUNT { } 를 쓴다. 중괄호 안에는 (1)에서 쓴 관계 패턴을 그대로 넣되 바깥의 m 을 그 안에서 쓰고, 비교값은 (1)의 조건과 같다.
6. RETURN m.name AS 메뉴 를 이름순으로 정렬해 받은 뒤, 파이썬에서 이름만 뽑아 sub_names 에 담는다.
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
assert len(rows) == 3, '3행이어야 합니다. 조건을 WITH 뒤 WHERE 에 걸었는지 확인하세요'
assert rows[0]['메뉴'] == '연어초밥' and rows[0]['주문건수'] == 3, '1위가 다릅니다. 수량 합이 아니라 주문 건수(count)를 셌는지 확인하세요'
assert all(r['주문건수'] >= 2 for r in rows), '2건 미만인 메뉴가 남아 있습니다'
assert rows == sorted(rows, key=lambda r: (-r['주문건수'], r['메뉴'])), '정렬을 확인하세요: 주문건수 내림차순, 같으면 메뉴 이름 오름차순'
# (2) 두 길이 같은 답을 내야 한다
assert sorted(sub_names) == sorted(r['메뉴'] for r in rows), 'COUNT { } 로 낸 목록이 (1)과 다릅니다. 중괄호 안 패턴이 같은 관계를 가리키는지, 조건이 >= 2 인지 확인하세요'
assert sub_names == sorted(sub_names), 'sub_names 는 메뉴 이름 오름차순이어야 합니다'
print('✅ 통과!')

## 9. 메뉴별 총 주문 수량을 속성으로 저장 (SET)
**배경**: 자주 쓰는 총 주문 수량을 매번 계산하지 않도록 메뉴 속성으로 저장합니다.

**요구사항**:
- `(:Customer)-[x:ORDERED]->(m:Menu)` 로 잡아 메뉴별 `sum(x.qty)` 를 집계하고, 각 `Menu` 노드의 파생 속성 **`total_ordered`** 로 저장하세요(`WITH` 로 집계 후 `SET`).
- 저장한 뒤, `m.name`·`m.total_ordered` 를 **`total_ordered` 내림차순**(같으면 이름 오름차순)으로 조회해 **`rows`** 에 담으세요. 별칭은 **`메뉴`**·**`총수량`**.
- **주문이 한 번도 없어 `total_ordered` 가 생기지 않은 메뉴는 제외**하세요(전체 7개 중 **6개**가 남습니다). 속성이 없는 노드는 그 값을 읽으면 `null` 이 되고, `null` 이 섞이면 정렬이 엉킵니다.

**예시**: 저장 후 `연어초밥` 의 `total_ordered` 는 **4** 이고, `rows` 는 **6행**입니다.

<details><summary>힌트</summary>

```text
접근방법:
- 첫 쿼리에서 메뉴별 sum(qty)를 WITH 로 집계해 SET 으로 저장하고, 둘째 쿼리에서 그 속성으로 조회한다.

세부구현:
1. 첫 run_cypher: 고객→메뉴(ORDERED)를 MATCH → WITH 로 메뉴와 sum(qty) 집계 → SET 으로 그 값을 메뉴의 total_ordered 속성에 저장한다.
2. 둘째 run_cypher: Menu 를 MATCH 하되 total_ordered 가 있는(IS NOT NULL) 것만, RETURN 에 메뉴 이름·total_ordered 를 별칭(메뉴·총수량)으로 두고 총수량 내림차순(같으면 메뉴순) 정렬한다.
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
# 파이썬 변수뿐 아니라 DB 를 다시 조회해 SET 이 실제로 저장됐는지 확인한다
assert run_cypher("MATCH (m:Menu {name:'연어초밥'}) RETURN m.total_ordered AS v")[0]['v'] == 4, \
    '연어초밥 노드에 total_ordered 가 저장되지 않았습니다. 조회만 하지 말고 SET 까지 실행했는지 확인하세요'
assert run_cypher("MATCH (m:Menu) WHERE m.total_ordered IS NOT NULL "
                  "RETURN count(m) AS n")[0]['n'] == 6
assert len(rows) == 6, '6행이어야 합니다. 주문이 없어 total_ordered 가 없는 메뉴를 제외했는지 확인하세요'
sushi = [r for r in rows if r['메뉴'] == '연어초밥'][0]
assert sushi['총수량'] == 4, '연어초밥의 총수량이 다릅니다. sum(x.qty) 로 집계했는지 확인하세요'
assert rows == sorted(rows, key=lambda r: (-r['총수량'], r['메뉴'])), '정렬을 확인하세요: 총수량 내림차순, 같으면 메뉴 이름 오름차순'
print('✅ 통과!')

## 10. 식당 분류를 구분 라벨로 바꾸기 (CASE 단순형)
**배경**: 메뉴를 식당의 `category` 값에 따라 보기 좋은 **구분 라벨**로 묶어 보여 줍니다. 값이 정해진 몇 가지 중 하나인지만 따지므로, 비교 대상을 `CASE` 뒤에 한 번만 적는 **단순형**이 알맞습니다.

**요구사항**:
- `(rt:Restaurant)-[:SERVES]->(m:Menu)` 에서 식당의 `category` 를 아래 표대로 라벨로 바꾸고, 그 라벨별 **메뉴 수**를 세어 **`rows`** 에 담으세요. 별칭은 **`구분`**·**`메뉴수`**.

| `category` 값 | `구분` 라벨 |
|---|---|
| `한식` | `국내식` |
| `일식` | `아시아식` |
| `양식` | `서양식` |
| 그 밖의 값 | `기타` |

- **구분 이름 오름차순**으로 정렬하세요.

**예시**: 결과는 **3행**이고 각 행에 `구분`·`메뉴수` 가 담깁니다. 모든 행의 `메뉴수` 를 더하면 전체 메뉴 수(**7**)와 같습니다(빠지는 메뉴가 없어야 합니다).

<details><summary>힌트</summary>

```text
접근방법:
- 식당과 메뉴를 SERVES 로 잇고, 식당 분류값을 표대로 라벨로 바꾼다. 그 라벨은 집계가 아니므로 그대로 그룹핑 키가 되어 라벨별로 메뉴가 세어진다.

세부구현:
1. MATCH 로 식당→메뉴(SERVES) 패턴을 잡는다.
2. RETURN 에 라벨 변환식을 별칭 구분 으로 둔다. 단순형이므로 비교할 값(식당의 category)을 CASE 바로 뒤에 한 번만 적고, 이어서 값마다 WHEN 을 쓴다. 표에 없는 값은 ELSE 로 받는다.
3. 같은 RETURN 에 메뉴를 세는 집계를 별칭 메뉴수 로 함께 둔다.
4. ORDER BY 구분 으로 정렬한다.
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
assert len(rows) == 3, '3행이어야 합니다. 라벨이 그룹핑 키가 됐는지 확인하세요'
assert {r['구분'] for r in rows} == {'국내식', '서양식', '아시아식'}, \
    '구분 라벨이 다릅니다. 표대로 한식은 국내식, 일식은 아시아식, 양식은 서양식 인지 확인하세요'
assert {r['구분']: r['메뉴수'] for r in rows} == {'국내식': 2, '서양식': 3, '아시아식': 2}, \
    '라벨별 메뉴 수가 다릅니다. 라벨을 그룹핑 키로 두고 메뉴를 셌는지 확인하세요'
assert rows == sorted(rows, key=lambda r: r['구분']), '구분 이름 오름차순으로 정렬하세요'
assert sum(r['메뉴수'] for r in rows) == 7, '라벨별 메뉴 수의 합이 전체와 다릅니다. ELSE 로 남는 값을 받았는지 확인하세요'
# DB 로 검산: '서양식' 은 category 가 '양식' 인 식당의 메뉴 수와 같아야 한다
west = run_cypher("MATCH (rt:Restaurant {category: '양식'})-[:SERVES]->(m:Menu) "
                  "RETURN count(m) AS n")[0]['n']
assert [r for r in rows if r['구분'] == '서양식'][0]['메뉴수'] == west, \
    '서양식 메뉴 수가 DB 를 다시 센 값과 다릅니다. 예시 값을 옮겨 적지 말고 직접 조회하세요'
print('✅ 통과!')

## 11. 고객 이름에 인덱스 만들기
**배경**: 고객을 이름으로 자주 찾는다면 `Customer.name` 에 인덱스를 둡니다.

**요구사항**:
- `Customer.name` 에 인덱스 **`customer_name_idx`** 를 만들고(`IF NOT EXISTS`), `db.awaitIndexes()` 로 기다린 뒤, `SHOW INDEXES` 로 인덱스 목록을 조회해 **`rows`** 에 담으세요(LOOKUP 제외). 별칭은 **`name`**(인덱스 이름).

**예시**: `rows` 의 `name` 값들에 **`customer_name_idx`** 가 들어 있습니다.

<details><summary>힌트</summary>

```text
접근방법:
- 교안의 인덱스 3단계: CREATE INDEX 로 만들고 → db.awaitIndexes 로 기다린 뒤 → SHOW INDEXES 로 확인.

세부구현:
1. CREATE INDEX 로 Customer 의 name 속성에 인덱스(이름 customer_name_idx, IF NOT EXISTS)를 만든다.
2. CALL db.awaitIndexes() 로 생성이 끝날 때까지 기다린다.
3. SHOW INDEXES 로 LOOKUP 을 제외한 이름 목록을 조회해 rows 에 담고, name 값만 뽑는다.
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
names = [r['name'] for r in rows]
# 지문이 요구한 대로 LOOKUP 을 걸러 냈는지, 이름이 아니라 DB 가 알려 주는 종류로 확인한다
kinds = {r['name']: r['type'] for r in
         run_cypher("SHOW INDEXES YIELD name, type RETURN name, type")}
assert all(kinds.get(n) != 'LOOKUP' for n in names), \
    'LOOKUP 인덱스가 섞여 있습니다. SHOW INDEXES 에 종류가 LOOKUP 이 아닌 것만 남기세요'
assert 'customer_name_idx' in names, '인덱스 이름을 customer_name_idx 로 만들었는지, SHOW INDEXES 결과를 rows 에 담았는지 확인하세요'
# DB 를 다시 조회해 인덱스가 실제로 만들어졌고 사용 가능한 상태인지 확인한다
live = run_cypher("SHOW INDEXES YIELD name, state RETURN name, state")
found = [r for r in live if r['name'] == 'customer_name_idx']
assert len(found) == 1 and found[0]['state'] == 'ONLINE', '인덱스 상태가 ONLINE 이 아닙니다. CALL db.awaitIndexes() 로 기다렸는지 확인하세요'
print('✅ 통과!')

## 12. 평균만 보면 놓치는 것 (중앙값과 분모)
**배경**: 4번에서 평균을, 5번에서 최고가·최저가를 냈습니다. 그런데 평균 하나로는 **값이 어떻게 퍼져 있는지**를 알 수 없습니다. 그리고 평균에는 함정이 하나 더 있습니다. 집계 함수는 **빠진 값을 건너뛰기** 때문에 분모가 조용히 달라집니다. 두 가지를 한 문제에서 확인합니다.

**요구사항**:
- (1) 모든 `Menu` 의 `price` 로 **평균가격**(소수 둘째 자리까지 `round`)·**중앙값**(`percentileCont(m.price, 0.5)`)·**최고가**(`max`)를 한 줄로 내어 **`rows`** 에 담으세요. 별칭은 **`평균가격`**·**`중앙값`**·**`최고가`**.
- (2) 메뉴별 총 주문 수량(`sum(x.qty)`)의 평균을 **두 가지 방법**으로 내세요. 별칭은 둘 다 **`메뉴수`**·**`평균수량`**·**`합계`** 이고, `평균수량` 은 소수 둘째 자리까지 `round` 합니다.
  - **`strict`**: `(:Customer)-[x:ORDERED]->(m:Menu)` 로 잡아 **주문이 있는 메뉴만** 집계합니다.
  - **`loose`**: `MATCH (m:Menu)` 로 전체 메뉴를 잡고 `OPTIONAL MATCH` 로 주문을 이어 **주문이 없는 메뉴도 0 으로** 함께 집계합니다.
  - 두 변수에는 `run_cypher` 결과의 **첫 행(dict)** 을 담으세요(`[0]` 까지 꺼내 둡니다).

**예시**: (1) `평균가격` 은 **11714.29**, `중앙값` 은 **12000** 입니다. (2) `strict['평균수량']` 은 **2.33**(메뉴수 6), `loose['평균수량']` 은 **2.0**(메뉴수 7) 이고, **`합계` 는 둘 다 14** 로 같습니다.

<details><summary>힌트</summary>

```text
접근방법:
- (1) 다른 집계 함수와 똑같이 RETURN 에 나란히 쓴다. percentileCont 의 둘째 인자 0.5 가 중앙값이다.
- (2) 같은 물음을 MATCH 로 한 번, MATCH + OPTIONAL MATCH 로 한 번 낸다. 분자(합계)는 같고 분모(메뉴수)만 달라진다.

세부구현:
1. MATCH 로 Menu 를 잡고 RETURN 에 round(avg(...), 2)·percentileCont(..., 0.5)·max(...) 를 요구사항의 별칭으로 둔다.
2. strict: 주문 관계를 타고 들어가 WITH 로 메뉴별 수량을 먼저 낸 뒤, count·round(avg)·sum 을 낸다.
3. loose: MATCH (m:Menu) 로 전체를 잡고 OPTIONAL MATCH 로 주문을 이은 뒤 2번과 같은 모양으로 낸다.
4. 두 결과 모두 [0] 으로 첫 행을 꺼내 strict·loose 에 담는다.
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
assert abs(rows[0]['평균가격'] - 11714.29) < 1e-6, '평균가격이 다릅니다. round(avg(m.price), 2) 로 소수 둘째 자리까지 줄였는지 확인하세요'
assert rows[0]['중앙값'] == 12000, '중앙값이 다릅니다. percentileCont(m.price, 0.5) 를 썼는지 확인하세요'
assert rows[0]['최고가'] == 15000, \
    '최고가가 다릅니다. max(m.price) 를 함께 냈는지 확인하세요'
assert strict['메뉴수'] == 6, 'strict 는 주문이 있는 메뉴만 세야 합니다(MATCH 로만 이었는지 확인하세요)'
assert loose['메뉴수'] == 7, 'loose 는 전체 메뉴를 세야 합니다(MATCH (m:Menu) 에 OPTIONAL MATCH 를 이었는지 확인하세요)'
assert abs(strict['평균수량'] - 2.33) < 1e-6 and abs(loose['평균수량'] - 2.0) < 1e-6, '평균수량이 다릅니다. 메뉴별 sum(x.qty) 를 WITH 로 먼저 낸 뒤 round(avg(...), 2) 로 소수 둘째 자리까지 줄였는지 확인하세요'
assert strict['합계'] == loose['합계'], '합계는 두 방법이 같아야 합니다. 분자는 같고 분모만 달라지는 것이 이 문제의 요점입니다'
# 지문이 여섯 값을 다 알려 주므로 손으로 적어도 위까지는 통과한다. DB 를 다시 조회해 대조한다
# (1) 의 세 값도 다시 조회해 대조한다
again1 = run_cypher("MATCH (m:Menu) "
                    "RETURN round(avg(m.price), 2) AS 평균가격, "
                    "       percentileCont(m.price, 0.5) AS 중앙값, "
                    "       max(m.price) AS 최고가")[0]
assert (rows[0]['평균가격'], rows[0]['중앙값'], rows[0]['최고가']) == \
       (again1['평균가격'], again1['중앙값'], again1['최고가']), \
    '세 값이 DB 를 다시 조회한 값과 다릅니다. 예시 값을 옮겨 적지 말고 직접 조회하세요'
again = run_cypher("MATCH (m:Menu) OPTIONAL MATCH (:Customer)-[x:ORDERED]->(m) "
                   "WITH m, sum(x.qty) AS 수량 "
                   "RETURN count(m) AS 메뉴수, sum(수량) AS 합계")[0]
assert loose['메뉴수'] == again['메뉴수'] and loose['합계'] == again['합계'], \
    'loose 값이 DB 를 다시 조회한 값과 다릅니다. 예시 값을 옮겨 적지 말고 직접 조회하세요'
print('✅ 통과!')

## 13. 값이 없는 자리를 0 으로 메우기 (coalesce)
**배경**: 9번에서 메뉴마다 `total_ordered` 를 저장했습니다. 그런데 **주문이 한 번도 없던 `알리오올리오` 에는 그 속성이 아예 붙지 않았습니다.** 없는 속성을 읽으면 `null` 이라, 그대로 화면에 내면 빈칸이 되고 정렬도 엉킵니다. 이럴 때 쓰는 것이 `coalesce` 입니다. (**9번을 먼저 풀어야 합니다.**)

**쓸 함수**: `coalesce(값1, 값2, ...)` 는 **앞에서부터 처음으로 `null` 이 아닌 값**을 돌려줍니다. `coalesce(m.total_ordered, 0)` 이라고 적으면 속성이 있으면 그 값이, 없으면 `0` 이 나옵니다. (교안_01 7-2 스칼라 함수에서 본 그 함수입니다.)

**요구사항**:
- 9번과 달리 **주문이 없는 메뉴도 빼지 말고 전체 `Menu` 를 한 행씩** 내세요. `total_ordered` 가 없는 메뉴는 **`coalesce` 로 `0` 을 채워** **`rows`** 에 담으세요. 별칭은 **`메뉴`**·**`총수량`**.
- **총수량 내림차순, 같으면 메뉴 이름 오름차순**으로 정렬하세요.

**예시**: `rows` 는 **7행**이고, 맨 아래가 `알리오올리오`(총수량 **0**)입니다. 총수량을 모두 더하면 **14** 로 9번의 합계와 같습니다.

<details><summary>힌트</summary>

```text
접근방법:
- 관계를 타고 들어가지 말고 Menu 만 잡는다. 관계를 타면 주문 없는 메뉴가 또 사라진다.
- 없는 속성은 null 이므로, 읽는 자리를 coalesce 로 감싸 0 을 채운다.

세부구현:
1. MATCH 로 Menu 만 잡는다(ORDERED 관계를 쓰지 않는다).
2. RETURN 에 메뉴 이름과 coalesce(속성, 0) 을 요구사항의 별칭(메뉴·총수량)으로 둔다.
3. ORDER BY 로 총수량 내림차순, 같으면 메뉴 이름 오름차순으로 정렬한다.
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
assert len(rows) == 7, '7행이어야 합니다. 관계를 타지 말고 Menu 만 잡았는지 확인하세요'
assert [(r['메뉴'], r['총수량']) for r in rows] == [('연어초밥', 4), ('비빔밥', 3), ('까르보나라', 2), ('마르게리타', 2), ('불고기', 2), ('우동', 1), ('알리오올리오', 0)], '값이나 정렬이 다릅니다. 총수량 내림차순(같으면 메뉴 이름 오름차순)인지 확인하세요'
assert [r for r in rows if r['메뉴'] == '알리오올리오'][0]['총수량'] == 0, '알리오올리오 의 총수량이 0 이 아닙니다. coalesce 로 null 을 0 으로 채웠는지 확인하세요'
assert sum(r['총수량'] for r in rows) == 14, '총수량의 합이 9번의 합계와 다릅니다'
print('✅ 통과!')

---
수고했어요! LV1 에서 count·sum·avg·min/max·중앙값·collect·WITH 필터·`COUNT { }`·SET·CASE(단순형)·인덱스·`coalesce` 를 **하나씩** 익혔습니다. LV2 에서는 이것들을 **조합**해 차트 랭킹·UNIQUE 제약과 조건형 CASE 를 다룹니다.